# Red Scare results
Run from `RedScare2026`. Each algorithm saves one CSV in `results/` and prints
runtime totals and averages by graph family. Timing excludes graph loading.
The last cell combines the saved files.


In [ ]:
from pathlib import Path
from time import perf_counter
from collections import defaultdict
from shutil import copyfileobj
import csv

files = sorted(Path('data').glob('*.txt'))
results_dir = Path('results')
results_dir.mkdir(exist_ok=True)
columns = ['algorithm', 'file', 'graph_type', 'result', 'runtime_seconds']
step_files = {
    'none': '01_none.csv',
    'some': '02_some.csv',
    'many': '03_many.csv',
    'few': '04_few.csv',
    'alternate': '05_alternate.csv',
}
graph_types = {
    file: '-'.join(file.stem.split('-')[:2]) if file.stem.startswith('wall-')
    else file.stem.split('-')[0]
    for file in files
}


def save_results(name, rows):
    with (results_dir / step_files[name]).open('w', newline='') as output:
        writer = csv.writer(output)
        writer.writerow(columns)
        writer.writerows(rows)


def run_algorithm(name, reader, solver):
    rows = []
    timings = defaultdict(list)
    for file in files:
        graph, s, t, red = reader(file)
        start = perf_counter()
        result = solver(graph, s, t, red)
        elapsed = perf_counter() - start
        rows.append([name, file.name, graph_types[file], result, elapsed])
        timings[graph_types[file]].append(elapsed)

    save_results(name, rows)
    for graph_type, times in sorted(timings.items()):
        print(f'{graph_type:<14} {len(times):>3} graphs  total {sum(times):.4f}s  avg {sum(times) / len(times):.4f}s')


## None

In [ ]:
from NoneProblem import read_graph_none, bfs_none


def solve_none(graph, s, t, red):
    path = bfs_none(graph, s, t)
    return len(path) if path else -1


run_algorithm('none', read_graph_none, solve_none)


## Some

In [ ]:
from SomeProblem import read_graph_some, some_function

run_algorithm('some', read_graph_some, some_function)


## Many
No solver is implemented yet; save a placeholder for each graph.

In [ ]:
save_results('many', [
    ['many', file.name, graph_types[file], 'not_implemented', '']
    for file in files
])
print('Many: no solver implemented yet.')


## Few

In [ ]:
from FewProblem import read_graph_few, dijkstra_few


def solve_few(graph, s, t, red):
    result = dijkstra_few(graph, s, t, red)
    return result if type(result) is int else -1


run_algorithm('few', read_graph_few, solve_few)


## Alternate

In [ ]:
from AlternateProblem import read_graph_alternating, bfs_alternating

run_algorithm('alternate', read_graph_alternating,
              lambda graph, s, t, red: bfs_alternating(graph, s, t))


## Combine saved results

In [ ]:
with (results_dir / 'combined_results.csv').open('w', newline='') as combined:
    for index, filename in enumerate(step_files.values()):
        with (results_dir / filename).open(newline='') as saved:
            header = saved.readline()
            if index == 0:
                combined.write(header)
            copyfileobj(saved, combined)

print('Saved results/combined_results.csv')
